# Agile Bioacoustic Modeling with SongSpace

This tutorial reproduces the core iterative workflow from the Hoplite agile modeling notebook, but uses `SongSpace` to manage datasets and shallow classifiers.

## Run this tutorial

If running in Colab, uncomment the installation line below.

In [1]:
# if 'google.colab' in str(get_ipython()):
#     %pip install "opensoundscape==0.12.1" "bioacoustics-model-zoo==0.12.0"

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

import bioacoustics_model_zoo as bmz

from opensoundscape.annotations import BoxedAnnotations
from opensoundscape.vector_database import load_or_create_hoplite_usearch_db
from opensoundscape.ml.song_space import SongSpace
from opensoundscape.ml.shallow_classifier import select_from_hoplite
from opensoundscape.visualization import annotate, inspect

## Prepare labels

This uses the same _Rana sierrae_ example files as the agile Hoplite tutorial.

In [3]:
dataset_path = Path("./rana_sierrae_2022/")
audio_and_raven_files = pd.read_csv(dataset_path / "audio_and_raven_files.csv")
audio_and_raven_files["audio"] = audio_and_raven_files["audio"].apply(
    lambda x: str(dataset_path / x)
)
audio_and_raven_files["raven"] = audio_and_raven_files["raven"].apply(
    lambda x: str(dataset_path / x)
)

annotations = BoxedAnnotations.from_raven_files(
    raven_files=audio_and_raven_files["raven"],
    audio_files=audio_and_raven_files["audio"],
    annotation_column="annotation",
)

labels = annotations.clip_labels(clip_duration=3, min_label_overlap=0.2)

/Users/SML161/opensoundscape/opensoundscape/annotations.py:347: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_annotations_df = pd.concat(all_file_dfs).reset_index(drop=True)


In [4]:
target_source_class = "C"
target_model_class = "RanaSierrae_C"

binary_labels = labels[[target_source_class]].rename(
    columns={target_source_class: target_model_class}
)
seed_train, validation = train_test_split(binary_labels, test_size=0.5, random_state=0)

# Create an unlabeled pool from the remaining clips (NaN labels are treated as weak negatives).
pool = binary_labels.drop(index=seed_train.index).copy()
pool[target_model_class] = np.nan

print("seed_train:", seed_train.shape)
print("validation:", validation.shape)
print("pool:", pool.shape)

seed_train: (1344, 1)
validation: (1344, 1)
pool: (1344, 1)


## Build database and SongSpace

In [5]:
ss = SongSpace("./ssdb")

downloading model from URL...
Connecting to existing db at songspace_agile_db
Connected database has 1,856 embeddings from 662 files.


In [ ]:
# Embed and register datasets in SongSpace.
ss.ingest_audio(
    seed_train.sample(100),
    dataset_name="round1_train",
    batch_size=32,
)
ss.ingest_audio(
    validation.sample(100),
    dataset_name="validation",
    allow_training=False,
    batch_size=32,
)
ss.ingest_audio(
    pool.sample(100),
    dataset_name="pool_unlabeled",
    batch_size=32,
)

ss.list_datasets()

all samples already have embeddings in the database
embedding 63 new windows to database


0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.


  0%|          | 0/2 [00:00<?, ?it/s]

## Train first classifier

In [ ]:
ss

In [ ]:
df, windows = ss.get_dataset("round1_train")
df

,,,RanaSierrae_C
file,start_time,end_time,


In [19]:
clf_round1 = ss.fit_classifier(
    classes=[target_model_class],
    train_datasets=["round1_train"],
    validation_dataset="validation",
    weak_negatives_proportion=1.0,
    weak_negatives_weight=0.01,
    steps=100,
    batch_size=128,
    validation_interval=25,
    logging_interval=25,
)
clf_round1.val_metrics

adding 0 weak negatives to the training data
training classifier for 1 classes with 0 training samples and 79 validation samples
Finding matching window IDs for samples in label_df...
Zero samples passed to _find_matching_window_ids


ValueError: num_samples should be a positive integer value, but got num_samples=0

save the classifier in the SoundScape, if we like it enough

In [11]:
ss.add_classifier("rana_round1", clf_round1)

evaluate it

In [14]:
round1_metrics = ss.evaluate("rana_round1", "validation")
round1_metrics

  0%|          | 0/1 [00:00<?, ?it/s]

{'RanaSierrae_C': {'average_precision': 0.14806134434041412,
  'roc_auc': 0.42162162162162165},
 'macro_average_precision': np.float64(0.14806134434041412),
 'macro_roc_auc': np.float64(0.42162162162162165)}

## Active learning round: review high-scoring candidates

In [15]:
pool_scores = ss.predict_on_dataset(
    classifier_name="rana_round1", dataset_name="pool_unlabeled"
)
topk = pool_scores.nlargest(20, target_model_class).reset_index()

# Review and annotate interactively.
_ = annotate(
    topk, bandpass_range=(0, 2500), annotation_buttons=["Accept", "Reject"], N=20
)
topk.head()

  0%|          | 0/1 [00:00<?, ?it/s]

GridBox(children=(VBox(children=(HTML(value='<img src="data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAJsAAACaC…

,file,start_time,end_time,RanaSierrae_C,Accept,Reject
0,rana_sierrae_2022/mp3/sine2022a_MSD-0558_20220...,9.0,12.0,-5.832896,None,None
1,rana_sierrae_2022/mp3/sine2022a_MSD-0558_20220...,9.0,12.0,-5.843591,None,None
2,rana_sierrae_2022/mp3/sine2022a_MSD-0558_20220...,9.0,12.0,-5.891069,None,None
3,rana_sierrae_2022/mp3/sine2022a_MSD-0558_20220...,9.0,12.0,-5.904296,None,None
4,rana_sierrae_2022/mp3/sine2022a_MSD-0558_20220...,9.0,12.0,-5.974036,None,None


format labels

In [ ]:
new_pos = topk[topk["Accept"] == True][["file", "start_time", "end_time"]].copy()
new_pos[target_model_class] = 1

new_neg = topk[topk["Reject"] == True][["file", "start_time", "end_time"]].copy()
new_neg[target_model_class] = 0

round2_train = (
    pd.concat([new_pos, new_neg], ignore_index=True)
    .drop_duplicates()
    .set_index(["file", "start_time", "end_time"])[[target_model_class]]
)
round2_train.shape

## Retrain with expanded labels

In [ ]:
ss.ingest_audio(
    round2_train,
    dataset_name="round2_train",
    deployment="rana_sierrae_2022",
    batch_size=32,
    num_workers=0,
)

clf_round2 = ss.fit_classifier(
    classes=[target_model_class],
    train_datasets=["round1_train", "round2_train", "pool_unlabeled"],
    validation_dataset="validation",
    weak_negatives_proportion=1.0,
    weak_negatives_weight=0.01,
    steps=300,
    batch_size=128,
    validation_interval=25,
    logging_interval=25,
)
ss.add_classifier("rana_round2", clf_round2)

round2_metrics = ss.evaluate("rana_round2", "validation")
round2_metrics

## Select clips for manual verification

Use stratified or thresholded selection from the full embedded database.

In [ ]:
selected = select_from_hoplite(
    db,
    ss.classifiers["rana_round2"],
    classes=[target_model_class],
    strategy="top_k",
    k=20,
    min_score=0,
)
selected.head()

In [ ]:
# Optional cleanup
# import shutil
# shutil.rmtree('./songspace_agile_db/')